# 11_03 · 로짓 재덤프 (`11_01` b128)

In [1]:
# flash-attn 프리빌트 휠(cu12·torch2.11·cp312) — 백본이 flash_attention_2를 요구
!wget -q "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install -q flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
# transformers 버전 pin — 로컬 .venv(5.13.0)와 같은 ModernBERT+FA2 regime 고정(colab-jobs.md)
!pip install -q "transformers==5.13.0" "accelerate>=1.1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 140.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os, sys, json, shutil

DRIVE = "/content/drive/MyDrive/patent_disc"        # Drive 프로젝트 루트(코드·산출물)

# 코드 반입 — Drive의 patent_train을 로컬 /content/src로 복사해 import(네트워크 I/O 회피).
shutil.copytree(f"{DRIVE}/src/patent_train", "/content/src/patent_train",
                dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
sys.path.insert(0, "/content/src")

import numpy as np
import torch
import transformers
import patent_train
from patent_train import TrainConfig, TrainingRunner

# 로드 경로·버전 확인 — 로컬 사본을 import했는지, regime pin이 걸렸는지
print("patent_train :", patent_train.__file__)   # /content/src/patent_train/__init__.py 여야
print("transformers :", transformers.__version__)
print("torch        :", torch.__version__, "| cuda:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

patent_train : /content/src/patent_train/__init__.py
transformers : 5.13.0
torch        : 2.11.0+cu128 | cuda: NVIDIA L4


In [4]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")   # Hub 모델·데이터셋 다운로드
os.environ["HF_HOME"] = "/content/.hf_cache"                        # 휘발 — 매 VM 새로 받는다
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [5]:
cfg = TrainConfig.for_inference(
    tag="modernbert-patent-len512-b128",
    checkpoint="ingyoun/A.X-patent-len512-b128",   # 11_01이 push한 모델(Hub)
    out_path="/content/output/redump",             # save_metrics 미사용 — 로짓은 아래 out_dir로 Drive 직결
    workspace="/content",
    max_len=512,
    eval_micro_batch=512,
    splits=("val", "test"),                        # train(201k행) 로드 회피
)
print("checkpoint:", cfg.checkpoint, "| splits:", cfg.splits, "| max_len:", cfg.max_len)

checkpoint: ingyoun/A.X-patent-len512-b128 | splits: ('val', 'test') | max_len: 512


## 로드 — 모델·데이터셋(Hub)

`load_data`(토크나이저+원본 val/test) → `prepare_data`(max_len 절단) → `load_model`(체크포인트 복원)을 단계별로 실행한다(11_01과 동일). 추론 경로는 on-disk prep 캐시를 우회한다.

In [6]:
runner = TrainingRunner(cfg)

In [7]:
runner.load_data()       # 토크나이저 + 원본(val/test) 로드
runner.data.raw

config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

...

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/201616 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11244 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/11132 [00:00<?, ? examples/s]

DatasetDict({
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11132
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11244
    })
})

In [8]:
runner.prepare_data()    # max_len 절단(추론 경로는 prep 캐시 우회)
runner.data.dataset

Map:   0%|          | 0/11132 [00:00<?, ? examples/s]

Map:   0%|          | 0/11244 [00:00<?, ? examples/s]

DatasetDict({
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11132
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11244
    })
})

In [9]:
runner.load_model()

config.json:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  598MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertForSequenceClassification is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`
[transformers] Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B", attn_implementation="flash_attention_2", dtype=torch.float16)`


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[model] ingyoun/A.X-patent-len512-b128


In [10]:
runner.build_trainer()   # wandb·early stop·save 없음, 순차 샘플러

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 재덤프 · 검증 (로짓 → Drive)

In [11]:
SSOT_TEST_MICRO = 0.8588              # 11_01 modernbert-patent-len512-b128_metrics.json
LOGIT_DIR = f"{DRIVE}/output"         # 0로짓을 Drive에 저장

for split in ("val", "test"):
    logits = runner.predict_logits(split, out_dir=LOGIT_DIR)   # 순차 샘플러 + 행 순서 assert + 지표 보관
    micro = runner.metrics[split]["test_micro_f1"]             # predict가 함께 낸 micro(prefix=test)
    print(f"[{split}] logits {logits.shape} → {LOGIT_DIR} · micro_f1 {micro:.4f}")

assert abs(runner.metrics["test"]["test_micro_f1"] - SSOT_TEST_MICRO) < 1e-3, runner.metrics["test"]
print(f"verify: test micro ≈ SSOT {SSOT_TEST_MICRO} — Hub 모델 정상 복원 · 순열 재발 없음")

[dump] /content/drive/MyDrive/patent_disc/output/logits_modernbert-patent-len512-b128_val.npy  shape=(11132, 188)
[val] logits (11132, 188) → /content/drive/MyDrive/patent_disc/output · micro_f1 0.8623


[dump] /content/drive/MyDrive/patent_disc/output/logits_modernbert-patent-len512-b128_test.npy  shape=(11244, 188)
[test] logits (11244, 188) → /content/drive/MyDrive/patent_disc/output · micro_f1 0.8588
verify: test micro ≈ SSOT 0.8588 — Hub 모델 정상 복원 · 순열 재발 없음
